# Interval Arithmetic

In his *Mathematica* notebook, Jerry Keiper distinguishes two ways a computer can track numerical uncertainty:

- **Range arithmetic**: intervals are implicit — each floating-point value   carries a precision count bounding the rounding error, but the bounds are   not visible in the value itself.
- **Interval arithmetic**: intervals are explicit — a value is a pair   `[lo, hi]` that *guarantees* the true result lies between the endpoints.   Arithmetic propagates these bounds conservatively.

Gallowglass computes over natural numbers, so we build explicit interval arithmetic over `Nat`. A sensor reading known to lie between 10 and 15 is `MkInterval 10 15`; adding a second reading in `[3, 6]` gives a guaranteed sum in `[13, 21]`.

This notebook builds an interval typeclass from scratch:

1. **Type** — define `Interval`
2. **Functions** — `iv_lo`, `iv_hi`, `iv_width`, `iv_contains`, `iv_add`, `iv_mul`
3. **Show instance** — human-readable `"[lo, hi]"` rendering
4. **Typeclass** — `IvArith` abstracts over anything that behaves like an interval
5. **Instance** — wire `Interval` into `IvArith`
6. **Constrained wrappers** — dispatch through the typeclass constraint
7. **Applications** — tracking bounds through multi-step computations

Assumes `02-typeclasses.ipynb` — constrained wrappers, instance syntax.

## Imports

Arithmetic primitives from `Core.Nat`; text utilities from `Core.Text`:

In [1]:
use Core.Nat unqualified { add, mul, sub, nat_lte }
use Core.Text { Show, show, show_nat, text_concat }

use Core.Nat
use Core.Text

## The `Interval` type

An interval is a lower bound and an upper bound. One constructor, 
two `Nat` fields:

In [2]:
type Interval =
  | MkInterval Nat Nat

type Interval = MkInterval

Extract the bounds with pattern matching:

In [3]:
let iv_lo : Interval → Nat
  = λ ii → match ii { | MkInterval ll _ → ll }

iv_lo : Notebook.Interval → Nat

In [4]:
let iv_hi : Interval → Nat
  = λ ii → match ii { | MkInterval _ hh → hh }

iv_hi : Notebook.Interval → Nat

## Width and containment

The **width** of an interval is `hi − lo`. `sub` is saturating (result is 0 when the subtrahend exceeds the minuend), so a degenerate point interval `MkInterval 5 5` safely yields width 0:

In [5]:
let iv_width : Interval → Nat
  = λ ii → match ii { | MkInterval ll hh → sub hh ll }

iv_width : Notebook.Interval → Nat

In [6]:
iv_width (MkInterval 3 7)

4

In [7]:
iv_width (MkInterval 5 5)

0

`iv_contains` tests whether `n ∈ [lo, hi]`. `nat_lte m n` returns `True` when `m ≤ n`; `if/then/else` dispatches on the `Bool` result:

In [8]:
let iv_contains : Interval → Nat → Bool
  = λ ii nn → match ii {
      | MkInterval ll hh →
          if nat_lte ll nn then nat_lte nn hh else False
    }

iv_contains : Notebook.Interval → Nat → Bool

In [9]:
iv_contains (MkInterval 3 7) 5

True

In [10]:
iv_contains (MkInterval 3 7) 10

False

## Interval arithmetic — standalone functions

If `a ∈ [la, ha]` and `b ∈ [lb, hb]`, then `a + b ∈ [la+lb, ha+hb]`. The result is the *tightest* interval guaranteed to contain every possible sum:

In [11]:
let iv_add : Interval → Interval → Interval
  = λ aa bb → match aa {
      | MkInterval la ha → match bb {
          | MkInterval lb hb → MkInterval (add la lb) (add ha hb)
        }
    }

iv_add : Notebook.Interval → Notebook.Interval → Notebook.Interval

For non-negative intervals, multiplication is monotone in both arguments — `[la·lb, ha·hb]` covers every possible product:

In [12]:
let iv_mul : Interval → Interval → Interval
  = λ aa bb → match aa {
      | MkInterval la ha → match bb {
          | MkInterval lb hb → MkInterval (mul la lb) (mul ha hb)
        }
    }

iv_mul : Notebook.Interval → Notebook.Interval → Notebook.Interval

Two intervals from Keiper's notebook — a ∈ [2, 3], b ∈ [4, 5]:

In [13]:
iv_add (MkInterval 2 3) (MkInterval 4 5)

MkInterval 6 8

In [14]:
iv_mul (MkInterval 2 3) (MkInterval 4 5)

MkInterval 8 15

The width of the sum equals the sum of the widths — uncertainty accumulates linearly under addition:

In [15]:
iv_width (iv_add (MkInterval 2 3) (MkInterval 4 5))

2

## `Show` instance

Give `Interval` a human-readable `"[lo, hi]"` rendering. `show_nat` converts a `Nat` to decimal text; `text_concat` joins `Text` values:

In [16]:
instance Show Interval {
  show = λ ii → match ii {
      | MkInterval ll hh →
          text_concat "["
            (text_concat (show_nat ll)
              (text_concat ", "
                (text_concat (show_nat hh) "]")))
    }
}

The dispatch wrapper — same pattern as lesson 02; `show` only fires through a `let` whose type carries the `Show` constraint:

In [17]:
let as_text : ∀ a. Show a => a → Text = λ xx → show xx

as_text : ∀ a. a → Text

In [18]:
as_text (MkInterval 3 7)

[3, 7]

## The `IvArith` typeclass

Abstract over the concrete type. Any type that implements `IvArith` exposes lower and upper bounds, supports addition and multiplication, tests containment, and reports its width.

The class method names mirror the standalone function names — the same pattern as `class Add a { add }` in `Core.Nat` alongside `let add : Nat → Nat → Nat`:

In [19]:
class IvArith i {
  iv_lo       : i → Nat
  iv_hi       : i → Nat
  iv_add      : i → i → i
  iv_mul      : i → i → i
  iv_contains : i → Nat → Bool
  iv_width    : i → Nat
}

## Instance for `Interval`

Wire the standalone functions into the instance. The pattern `iv_add = iv_add` assigns the standalone function to the class method — identical to `instance Add Nat { add = add }` in `Core.Nat`:

In [20]:
instance IvArith Interval {
  iv_lo       = iv_lo
  iv_hi       = iv_hi
  iv_add      = iv_add
  iv_mul      = iv_mul
  iv_contains = iv_contains
  iv_width    = iv_width
}

## Constrained wrappers

Dispatch fires through a `let` whose type carries the constraint. Define thin wrappers — these are the entry points callers use:

In [21]:
let add_iv : ∀ i. IvArith i => i → i → i
  = λ aa bb → iv_add aa bb

add_iv : ∀ i. i → i → i

In [22]:
let mul_iv : ∀ i. IvArith i => i → i → i
  = λ aa bb → iv_mul aa bb

mul_iv : ∀ i. i → i → i

In [23]:
let contains_iv : ∀ i. IvArith i => i → Nat → Bool
  = λ ii nn → iv_contains ii nn

contains_iv : ∀ i. i → Nat → Bool

In [24]:
let width_iv : ∀ i. IvArith i => i → Nat
  = λ ii → iv_width ii

width_iv : ∀ i. i → Nat

## Application: propagating measurement bounds

A pressure sensor reads between 10 and 15 PSI; a temperature sensor reads between 3 and 6. Track the guaranteed range of their sum:

In [25]:
let pressure : Interval = MkInterval 10 15
let temp     : Interval = MkInterval 3 6

pressure : Notebook.Interval
temp : Notebook.Interval

In [26]:
as_text (add_iv pressure temp)

[13, 21]

Is 20 guaranteed to be in the combined range? Is 22?

In [27]:
contains_iv (add_iv pressure temp) 20

True

In [28]:
contains_iv (add_iv pressure temp) 22

False

And the product:

In [29]:
as_text (mul_iv pressure temp)

[30, 90]

## Application: polynomial over an interval

Keiper's notebook evaluates Rump's pathological polynomial to demonstrate catastrophic cancellation in floating-point arithmetic. The Gallowglass analogue: evaluate `f(x) = 2·x² + 3·x` for `x ∈ [4, 6]` using interval arithmetic to bound the result.

A constant wrapped as a point interval:

In [30]:
let point : Nat → Interval = λ nn → MkInterval nn nn

point : Nat → Notebook.Interval

`x` squared via `mul_iv`:

In [31]:
let xx : Interval = MkInterval 4 6

xx : Notebook.Interval

In [32]:
let x_sq : Interval = mul_iv xx xx

x_sq : Notebook.Interval

In [33]:
as_text x_sq

[16, 36]

`2·x²` and `3·x`:

In [34]:
let term1 : Interval = mul_iv (point 2) x_sq

term1 : Notebook.Interval

In [35]:
let term2 : Interval = mul_iv (point 3) xx

term2 : Notebook.Interval

In [36]:
as_text term1

[32, 72]

In [37]:
as_text term2

[12, 18]

The guaranteed range of `f(x)`:

In [38]:
let fx : Interval = add_iv term1 term2

fx : Notebook.Interval

In [39]:
as_text fx

[44, 90]

Verify: at x=4, f(4) = 2·16 + 12 = 44. At x=6, f(6) = 2·36 + 18 = 90. Both should be contained in the computed interval:

In [40]:
contains_iv fx 44

True

In [41]:
contains_iv fx 90

True

## Width growth and accumulated uncertainty

Each arithmetic step can widen the interval. Starting from a step size known only to lie in [1, 2], after three additions:

In [42]:
let step : Interval = MkInterval 1 2

step : Notebook.Interval

In [43]:
let after3 : Interval = add_iv step (add_iv step step)

after3 : Notebook.Interval

In [44]:
as_text after3

[3, 6]

In [45]:
width_iv after3

3

Width 3 — each step contributed its full uncertainty of 1. An exact step size of 1 gives total 3; step size of 2 gives 6. Both lie inside `[3, 6]`. The interval is a *guarantee*, not a best guess.

Multiplication widens faster — `[1, 3]` squared:

In [46]:
let base_iv : Interval = MkInterval 1 3

base_iv : Notebook.Interval

In [47]:
as_text (mul_iv base_iv base_iv)

[1, 9]

In [48]:
width_iv (mul_iv base_iv base_iv)

8

## Fixed-point display

The `Show` instance above renders raw `Nat` values. If your intervals represent measurements with two decimal places — `110` meaning `1.10`, `90` meaning `0.90` — you want the display to reflect that.

The key idea: a single `scale` constant at the top of the notebook parameterises the renderer. Change `scale` from `100` to `1000` and you get three decimal places everywhere, with no other edits.

In [49]:
use Core.Nat unqualified { nat_lt, div_nat, mod_nat }

use Core.Nat

`scale` is the notebook-level fixed-point denominator. `100` gives two decimal places:

In [50]:
let scale : Nat = 100

scale : Nat

Three helper functions feed `show_fixed`.

`count_digits n` — number of decimal digits in `n` (0 counts as 1):

In [51]:
let count_digits : Nat → Nat
  = λ nn →
      if nat_lt nn 10
      then 1
      else add 1 (count_digits (div_nat nn 10))

count_digits : Nat → Nat

`frac_digits sc` — decimal places implied by scale `sc`. For `sc = 100`: `count_digits 100 = 3`, minus 1 = 2 places:

In [52]:
let frac_digits : Nat → Nat
  = λ sc → sub (count_digits sc) 1

frac_digits : Nat → Nat

`zeros n` — a `Text` of `n` zero characters, used to left-pad the fractional part:

In [53]:
let zeros : Nat → Text
  = λ nn → match nn {
      | 0  → ""
      | kk → text_concat "0" (zeros kk)
    }

zeros : Nat → Text

`show_fixed sc nn` renders `nn` as a decimal with the number of places implied by `sc`. The padding count is `frac_digits sc − count_digits frac`, saturating to 0:

In [54]:
let show_fixed : Nat → Nat → Text
  = λ sc nn →
      let whole = div_nat nn sc in
      let frac  = mod_nat nn sc in
      let pad   = sub (frac_digits sc) (count_digits frac) in
      text_concat (show_nat whole)
        (text_concat "."
          (text_concat (zeros pad) (show_nat frac)))

show_fixed : Nat → Nat → Text

With `scale = 100`, `90` renders as `"0.90"` and `110` as `"1.10"`:

In [55]:
show_fixed scale 90

0.90

In [56]:
show_fixed scale 110

1.10

In [57]:
show_fixed scale 5

0.05

Partially apply `show_fixed` to pin the scale — the result is an ordinary `Nat → Text` function:

In [58]:
let show_fp : Nat → Text = show_fixed scale

show_fp : Nat → Text

In [59]:
show_fp 90

0.90

In [60]:
show_fp 110

1.10

## `[lo, hi]` and `mid ± rad` in fixed-point

Both display formats are parameterised by scale the same way:

In [61]:
let show_iv_bounds : Nat → Interval → Text
  = λ sc ii → match ii {
      | MkInterval ll hh →
          text_concat "["
            (text_concat (show_fixed sc ll)
              (text_concat ", "
                (text_concat (show_fixed sc hh) "]")))
    }

show_iv_bounds : Nat → Notebook.Interval → Text

In [62]:
let bounds_fp : Interval → Text = show_iv_bounds scale

bounds_fp : Notebook.Interval → Text

In [63]:
bounds_fp (MkInterval 90 110)

[0.90, 1.10]

In [64]:
let show_iv_pm : Nat → Interval → Text
  = λ sc ii → match ii {
      | MkInterval ll hh →
          let mid = div_nat (add ll hh) 2 in
          let rad = div_nat (sub hh ll) 2 in
          text_concat (show_fixed sc mid)
            (text_concat " +- " (show_fixed sc rad))
    }

show_iv_pm : Nat → Notebook.Interval → Text

In [65]:
let pm_fp : Interval → Text = show_iv_pm scale

pm_fp : Notebook.Interval → Text

In [66]:
pm_fp (MkInterval 90 110)

1.00 +- 0.10

In [67]:
pm_fp (MkInterval 95 115)

1.05 +- 0.10

## Application: resistances in a circuit

Two resistors with manufacturing tolerance, measured in hundredths of an ohm. R₁ ∈ [1.30, 1.47] Ω, R₂ ∈ [2.20, 2.35] Ω — represented at scale 100 as:

In [68]:
let r1 : Interval = MkInterval 130 147
let r2 : Interval = MkInterval 220 235

r1 : Notebook.Interval
r2 : Notebook.Interval

Series resistance R₁ + R₂, displayed both ways:

In [69]:
bounds_fp (add_iv r1 r2)

[3.50, 3.82]

In [70]:
pm_fp (add_iv r1 r2)

3.66 +- 0.16

The `mid ± rad` form uses integer division, so results with odd total width round down. `[3.50, 3.82]` has width 32 (even), midpoint exactly 3.66, radius exactly 0.16 — no rounding here. An odd-width interval like `[3.50, 3.83]` would give `3.66 ± 0.16` (true midpoint 3.665, truncated).

## What's next

You've built a typeclass from scratch — type, functions, Show instance, class declaration, instance binding, constrained wrappers, fixed-point display, and applications.

- **Change `scale`** — swap `100` for `1000` at the top and re-run; `show_fp`, `bounds_fp`, and `pm_fp` all adapt with no other edits.
- **Subtraction** — `[la, ha] − [lb, hb] = [la−hb, ha−lb]` with saturating-sub guarding the lower bound; add `iv_sub` as a method to `IvArith` and implement it for `Interval`.
- **Second instance** — define `type TaggedInterval = | Tagged Interval Bool` pairing a bound with a confidence flag; implement `IvArith` for it and observe that `add_iv`, `mul_iv`, and `contains_iv` all work unchanged — that's the polymorphism payoff.
- **`spec/05-type-system.md`** — the full typeclass and constraint specification.
- **`doc/phrasebook.md`** — canonical Gallowglass patterns, including the typeclass dispatch idiom used throughout this notebook.